# Aula 22 — Agent Orchestration Runtimes

**Quando um loop agente simples deixa de ser suficiente?**

Nas Aulas 20 e 21, construímos duas capacidades importantes:

```text
Aula 20
→ Agent Harness
→ loop agente controlado
→ decisão / autorização / execução

Aula 21
→ planning explícito
→ dependências
→ validação
→ replanning
```

Agora damos um passo diferente:

> **tornar estado, transições, persistência, interrupções e retomada parte explícita do runtime.**

Regra da aula:

```text
framework
≠
orchestration runtime

graph
≠
melhor arquitetura

orchestration runtime
=
coordenação explícita
+ estado observável
+ transições
+ checkpoints
+ interrupt / resume
+ recovery
```

A pergunta central será:

> **Quando essa camada adicional conquista sua complexidade?**

## 1. Objetivos

Ao final, você deverá conseguir:

- distinguir **Agent Harness** de **Orchestration Runtime**;
- modelar uma execução como grafo de estado;
- identificar nodes, edges e conditional routing;
- tornar shared state explícito;
- implementar checkpointing;
- explicar durable execution;
- implementar interrupt / resume sob controle do runtime;
- distinguir retry, resume e recovery;
- distinguir **completed**, **interrupted**, **recoverable_error** e **failed**;
- comparar código procedural com um graph runtime usando métricas compatíveis;
- separar **evidência observada** de uma função contextual de utility;
- reconhecer quando um framework não é necessário;
- conectar uma política determinística a um Decision Provider governado;
- ler LangGraph como implementação de referência, não como definição do conceito.

## 📘 Glossário da aula

- **[Runtime de orquestração](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#runtime-de-orquestração)**
- **[Checkpoint](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#checkpoint)**
- **[Execução durável](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#execução-durável)**
- **[Interrupção e retomada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#interrupção-e-retomada)**

## 2. Harness, planner e orchestration runtime

Estas camadas podem coexistir, mas não são sinônimos.

```text
Decision Provider
→ propõe próxima ação

Agent Harness
→ controla o comportamento agente em runtime

Planner
→ representa plano, dependências e revisão

Orchestration Runtime
→ coordena estado, nodes, transitions,
  persistência, interrupt e resume
```

Um orchestration runtime pode fazer parte de um Agent Harness.

Da mesma forma:

```text
MCP
→ como capabilities são descobertas/acessadas

Orchestration Runtime
→ quando e em qual sequência essas capabilities executam
```

### 2.1 Arquitetura mínima, sem framework

Nesta aula usamos primeiro uma arquitetura vendor-neutral:

```text
Input / Goal
↓
Shared State
↓
Node
↓
Transition Policy
↓
Next Node
↓
Checkpoint
↓
Continue / Interrupt / Resume / Stop
```

Somente depois de compreender esse mecanismo faz sentido mapear conceitos para um framework como LangGraph.

In [ ]:
from dataclasses import dataclass, field, asdict
from copy import deepcopy
import pandas as pd

SCENARIOS = {
    "linear": {
        "request_valid": True,
        "lesson_ready": True,
        "revision_possible": False,
        "approval_required": False,
        "approval_granted": True,
        "transient_failures": 0,
        "process_loss_after_interrupt": False,
    },
    "branching": {
        "request_valid": True,
        "lesson_ready": False,
        "revision_possible": True,
        "approval_required": False,
        "approval_granted": True,
        "transient_failures": 0,
        "process_loss_after_interrupt": False,
    },
    "human_interrupt": {
        "request_valid": True,
        "lesson_ready": True,
        "revision_possible": False,
        "approval_required": True,
        "approval_granted": False,
        "transient_failures": 0,
        "process_loss_after_interrupt": False,
    },
    "transient_failure": {
        "request_valid": True,
        "lesson_ready": True,
        "revision_possible": False,
        "approval_required": False,
        "approval_granted": True,
        "transient_failures": 1,
        "process_loss_after_interrupt": False,
    },
    "durable_resume": {
        "request_valid": True,
        "lesson_ready": True,
        "revision_possible": False,
        "approval_required": True,
        "approval_granted": False,
        "transient_failures": 0,
        "process_loss_after_interrupt": True,
    },
}

print("Cenários:", list(SCENARIOS))

## 3. Shared state

O runtime precisa de um estado explícito.

Nesta versão, o estado é pequeno de propósito. O objetivo não é criar um framework genérico, mas tornar o mecanismo observável.

In [ ]:
@dataclass
class RuntimeState:
    scenario: str
    request_valid: bool = False
    lesson_ready: bool = False
    revision_possible: bool = False
    approval_required: bool = False
    approval_granted: bool = False
    process_loss_after_interrupt: bool = False
    release_note: bool = False
    revision_requested: bool = False
    published: bool = False
    node: str = "inspect"
    status: str = "running"
    reason: str = ""
    node_executions: int = 0
    transitions: int = 0
    tool_calls: int = 0
    checkpoints: int = 0
    retries: int = 0
    resumes: int = 0
    human_interventions: int = 0
    transient_failures_remaining: int = 0
    trace: list = field(default_factory=list)

def make_state(name):
    cfg = SCENARIOS[name]
    return RuntimeState(
        scenario=name,
        request_valid=cfg["request_valid"],
        lesson_ready=cfg["lesson_ready"],
        revision_possible=cfg["revision_possible"],
        approval_required=cfg["approval_required"],
        approval_granted=cfg["approval_granted"],
        process_loss_after_interrupt=cfg["process_loss_after_interrupt"],
        transient_failures_remaining=cfg["transient_failures"],
    )

display(pd.DataFrame([asdict(make_state("linear"))]).drop(columns=["trace"]))

## 4. Nodes e transições

Cada node executa uma responsabilidade pequena.

A política de transição decide o próximo node a partir do estado observado.

```text
node
→ transforma estado

transition policy
→ escolhe próximo node
```

Isso é diferente de esconder toda a lógica em um único `while`.

In [ ]:
def record(state, event):
    state.trace.append({
        "node": state.node,
        "event": event,
        "status": state.status,
        "reason": state.reason,
    })

def inspect_node(state):
    state.tool_calls += 1

    if not state.request_valid:
        state.reason = "invalid_request"
    elif not state.lesson_ready and state.revision_possible:
        state.reason = "needs_revision"
    elif not state.lesson_ready:
        state.reason = "not_ready"
    else:
        state.reason = ""

    record(state, "request inspected")
    return state

def prepare_node(state):
    state.release_note = True
    state.tool_calls += 1
    record(state, "release note prepared")
    return state

def request_revision_node(state):
    state.revision_requested = True
    state.tool_calls += 1
    state.status = "completed"
    state.reason = "revision_requested"
    record(state, "revision requested")
    return state

def approval_node(state):
    if not state.approval_granted:
        state.human_interventions += 1
        state.status = "interrupted"
        state.reason = "human_approval_required"
        record(state, "execution interrupted")
        return state

    state.reason = ""
    record(state, "approval observed")
    return state

def publish_node(state):
    state.tool_calls += 1

    if state.transient_failures_remaining > 0:
        state.transient_failures_remaining -= 1
        state.status = "recoverable_error"
        state.reason = "transient_publish_failure"
        record(state, "transient failure")
        return state

    state.published = True
    state.status = "completed"
    state.reason = "published"
    record(state, "published")
    return state

def complete_state(state, reason):
    state.status = "completed"
    state.reason = reason
    record(state, reason)
    return state

def fail_state(state, reason):
    state.status = "failed"
    state.reason = reason
    record(state, reason)
    return state

NODES = {
    "inspect": inspect_node,
    "prepare": prepare_node,
    "request_revision": request_revision_node,
    "approval": approval_node,
    "publish": publish_node,
}

In [ ]:
def next_node(state):
    if state.node == "inspect":
        if state.reason == "invalid_request":
            return None
        if state.reason == "needs_revision":
            return "request_revision"
        if state.reason == "not_ready":
            return None
        return "prepare"

    if state.node == "prepare":
        return "approval" if state.approval_required else "publish"

    if state.node == "approval":
        return "publish" if state.approval_granted else None

    if state.node in {"request_revision", "publish"}:
        return None

    return None

for name in ["linear", "branching", "human_interrupt"]:
    s = make_state(name)
    inspect_node(s)
    print(name, "->", next_node(s), "| reason:", s.reason or "-")

## 5. Checkpoint

Um checkpoint é um **snapshot persistido do estado**.

Aqui usamos um store em memória para manter o laboratório determinístico e sem side effects externos.

Em produção, a persistência poderia estar em arquivo, banco, serviço de estado ou storage distribuído — mas o conceito é o mesmo.

In [ ]:
CHECKPOINT_STORE = {}

def save_checkpoint(run_id, state):
    # O contador faz parte do próprio estado persistido.
    state.checkpoints += 1
    CHECKPOINT_STORE[run_id] = deepcopy(state)
    return deepcopy(state)

def load_checkpoint(run_id):
    return deepcopy(CHECKPOINT_STORE[run_id])

demo = make_state("linear")
save_checkpoint("demo", demo)
restored = load_checkpoint("demo")

print("Checkpoint salvo:", "demo" in CHECKPOINT_STORE)
print("Estado restaurado:", restored.scenario, restored.node, restored.status)
print("checkpoints persistidos:", restored.checkpoints)

## 6. Graph runtime mínimo

Agora conectamos:

```text
shared state
+ nodes
+ transition policy
+ checkpoint
+ interrupt / resume
+ retry / recovery
```

O runtime abaixo continua sendo Python simples. Isso é intencional:

> **primeiro o mecanismo; depois o framework.**

In [ ]:
def run_graph(state, run_id, checkpoint_each_node=True, max_transitions=20):
    while state.status == "running":
        if state.node not in NODES:
            fail_state(state, "invalid_node")
            break

        if state.transitions > max_transitions:
            fail_state(state, "transition_budget_exceeded")
            break

        state.node_executions += 1
        NODES[state.node](state)

        if checkpoint_each_node:
            save_checkpoint(run_id, state)

        if state.status in {"completed", "failed", "interrupted"}:
            break

        nxt = next_node(state)

        if state.status == "recoverable_error":
            # Recovery policy local: neste laboratório, a falha é retryable.
            state.retries += 1
            state.status = "running"
            state.reason = ""
            nxt = state.node  # retry do node atual

        if nxt is None:
            if state.reason in {"invalid_request", "not_ready"}:
                complete_state(state, state.reason)
            else:
                fail_state(state, state.reason or "no_valid_transition")
            break

        state.node = nxt
        state.transitions += 1

    return state

for name in ["linear", "branching", "transient_failure"]:
    s = run_graph(make_state(name), f"graph-{name}")
    print(
        name,
        "=>", s.status, s.reason,
        "nodes=", s.node_executions,
        "transitions=", s.transitions,
        "retries=", s.retries,
    )

## 7. Interrupt e resume

No cenário de aprovação humana, a execução deve **parar de forma controlada**, preservar o estado e continuar depois da decisão.

```text
run
→ interrupt
→ checkpoint
→ tempo passa
→ decisão humana
→ runtime carrega checkpoint
→ aplica evento externo
→ transition policy
→ resume
```

Resume não significa começar tudo novamente.

E há uma regra arquitetural importante:

> **quem decide de onde continuar é o runtime; o código de demonstração não deve saltar manualmente para o próximo node.**

In [ ]:
def resume_graph(run_id, **external_updates):
    state = load_checkpoint(run_id)

    if state.status != "interrupted":
        raise ValueError("Only interrupted runs can be resumed.")

    for key, value in external_updates.items():
        if not hasattr(state, key):
            raise KeyError(f"Unknown state field: {key}")
        setattr(state, key, value)

    state.status = "running"
    state.reason = ""
    state.resumes += 1

    # O checkpoint preservou o node que interrompeu.
    # A policy decide a continuação a partir do novo estado.
    nxt = next_node(state)
    if nxt is None:
        return fail_state(state, "resume_without_valid_transition")

    state.node = nxt
    state.transitions += 1
    return run_graph(state, run_id)

run_id = "human-demo"
state = run_graph(make_state("human_interrupt"), run_id)

print("Antes da aprovação:")
print(
    state.status, state.reason, state.node,
    "checkpoints=", state.checkpoints,
    "transitions=", state.transitions,
)

resumed = resume_graph(run_id, approval_granted=True)

print("\nDepois da aprovação:")
print(
    resumed.status, resumed.reason, resumed.node,
    "published=", resumed.published,
    "resumes=", resumed.resumes,
    "checkpoints=", resumed.checkpoints,
    "transitions=", resumed.transitions,
)

### 7.1 Durable execution

A execução é **durável** quando o progresso relevante sobrevive ao processo que a iniciou.

Nosso laboratório simula isso reconstruindo a execução apenas a partir do checkpoint.

O valor aparece quando reiniciar do zero seria caro, arriscado ou impossível.

In [ ]:
run_id = "durable-demo"
first_process = run_graph(make_state("durable_resume"), run_id)

print("Processo 1 terminou em:", first_process.status, first_process.reason)

# Simula perda completa do objeto em memória do processo original.
del first_process

# O processo 2 conhece somente o run_id e o evento externo de aprovação.
second_process = resume_graph(run_id, approval_granted=True)

print("Retomada em outro processo lógico:")
print("status:", second_process.status)
print("published:", second_process.published)
print("resumes:", second_process.resumes)
print("checkpoints:", second_process.checkpoints)

## 8. Retry, resume e recovery não são a mesma coisa

```text
retry
→ repetir uma operação que falhou

resume
→ continuar uma execução pausada a partir de estado preservado

recovery
→ restaurar uma condição executável depois de uma falha
  e então decidir se deve retry, resume, compensar ou parar
```

No cenário `transient_failure`, o runtime identifica uma falha recuperável e repete somente o node de publicação.

In [ ]:
s = run_graph(make_state("transient_failure"), "recovery-demo")
display(pd.DataFrame(s.trace))
print("retries:", s.retries)
print("published:", s.published)

## 9. Baseline procedural

Para comparar de forma justa, implementamos a **mesma tarefa** sem graph runtime.

O baseline também usa estado explícito, mas coordena tudo por código procedural.

Isso evita um falso contraste:

```text
graph runtime
vs
código procedural propositalmente ruim
```

A comparação deve ser entre duas implementações razoáveis.

In [ ]:
def new_procedural_state(name):
    cfg = deepcopy(SCENARIOS[name])
    return {
        "scenario": name,
        "cfg": cfg,
        "phase": "inspect",
        "status": "running",
        "reason": "",
        "published": False,
        "revision_requested": False,
        "node_executions": 0,
        "tool_calls": 0,
        "retries": 0,
        "resumes": 0,
        "human_interventions": 0,
        "reconstructed_steps": 0,
        "trace": [],
    }

def procedural_step(state, event, tool_call=False, reconstructed=False):
    state["node_executions"] += 1
    state["trace"].append(event)
    if tool_call:
        state["tool_calls"] += 1
    if reconstructed:
        state["reconstructed_steps"] += 1

def run_procedural(state):
    cfg = state["cfg"]

    while state["status"] == "running":
        if state["phase"] == "inspect":
            procedural_step(state, "inspect", tool_call=True)

            if not cfg["request_valid"]:
                state["status"] = "completed"
                state["reason"] = "invalid_request"
                break

            if not cfg["lesson_ready"] and cfg["revision_possible"]:
                state["phase"] = "request_revision"
                continue

            if not cfg["lesson_ready"]:
                state["status"] = "completed"
                state["reason"] = "not_ready"
                break

            state["phase"] = "prepare"
            continue

        if state["phase"] == "request_revision":
            procedural_step(state, "request_revision", tool_call=True)
            state["revision_requested"] = True
            state["status"] = "completed"
            state["reason"] = "revision_requested"
            break

        if state["phase"] == "prepare":
            procedural_step(state, "prepare", tool_call=True)
            state["phase"] = "approval" if cfg["approval_required"] else "publish"
            continue

        if state["phase"] == "approval":
            if not cfg["approval_granted"]:
                procedural_step(state, "approval_interrupt")
                state["human_interventions"] += 1
                state["status"] = "interrupted"
                state["reason"] = "human_approval_required"
                break
            state["phase"] = "publish"
            continue

        if state["phase"] == "publish":
            procedural_step(state, "publish", tool_call=True)
            if cfg["transient_failures"] > 0:
                cfg["transient_failures"] -= 1
                state["retries"] += 1
                continue
            state["published"] = True
            state["status"] = "completed"
            state["reason"] = "published"
            break

        state["status"] = "failed"
        state["reason"] = "invalid_phase"

    return state

def resume_procedural_same_process(state, approval_granted=True):
    if state["status"] != "interrupted":
        raise ValueError("Only interrupted runs can be resumed.")
    state["cfg"]["approval_granted"] = approval_granted
    state["status"] = "running"
    state["reason"] = ""
    state["resumes"] += 1
    return run_procedural(state)

def reconstruct_procedural_after_loss(name, prior_state, approval_granted=True):
    # Sem checkpoint, reconstruímos o estado por reexecução de passos observáveis.
    rebuilt = new_procedural_state(name)
    rebuilt["cfg"]["approval_granted"] = approval_granted
    rebuilt["resumes"] = prior_state["resumes"] + 1
    rebuilt["human_interventions"] = prior_state["human_interventions"]

    procedural_step(rebuilt, "reconstruct:inspect", tool_call=True, reconstructed=True)
    procedural_step(rebuilt, "reconstruct:prepare", tool_call=True, reconstructed=True)

    rebuilt["phase"] = "publish"
    return run_procedural(rebuilt)

for name in SCENARIOS:
    p = run_procedural(new_procedural_state(name))
    print(name, "=>", p["status"], p["reason"], "nodes=", p["node_executions"])

## 10. Comparative Evidence Lab

Agora comparamos as duas arquiteturas nos mesmos cinco cenários.

A primeira camada contém **somente medidas observadas**. Não há “vencedor” embutido na fórmula.

Medimos:

- task success;
- node executions — mesma unidade conceitual nas duas arquiteturas;
- graph transitions — métrica específica do runtime de grafo;
- tool calls;
- retries;
- resumes;
- human interventions;
- checkpoints;
- reconstructed steps após perda do processo.

> **Não misturamos nodes com edges e não agregamos pesos diferentes por arquitetura.**

Depois, em uma segunda camada, aplicaremos funções de utility **explicitamente declaradas e iguais para ambas as arquiteturas**.

In [ ]:
def graph_metrics(name):
    run_id = f"compare-{name}"
    s = run_graph(make_state(name), run_id)

    if s.status == "interrupted":
        if s.process_loss_after_interrupt:
            del s
        s = resume_graph(run_id, approval_granted=True)

    return {
        "scenario": name,
        "architecture": "graph_runtime",
        "success": bool(
            s.published
            or s.revision_requested
            or s.reason in {"invalid_request", "not_ready"}
        ),
        "outcome": s.reason,
        "published": s.published,
        "node_executions": s.node_executions,
        "graph_transitions": s.transitions,
        "tool_calls": s.tool_calls,
        "retries": s.retries,
        "resumes": s.resumes,
        "human_interventions": s.human_interventions,
        "checkpoints": s.checkpoints,
        "reconstructed_steps": 0,
        "process_loss": s.process_loss_after_interrupt,
    }

def procedural_metrics(name):
    first = run_procedural(new_procedural_state(name))
    process_loss = SCENARIOS[name]["process_loss_after_interrupt"]

    if first["status"] == "interrupted":
        if process_loss:
            final = reconstruct_procedural_after_loss(
                name,
                first,
                approval_granted=True,
            )
            # Soma o trabalho feito antes da perda ao trabalho após reconstrução.
            final["node_executions"] += first["node_executions"]
            final["tool_calls"] += first["tool_calls"]
        else:
            final = resume_procedural_same_process(
                first,
                approval_granted=True,
            )
    else:
        final = first

    return {
        "scenario": name,
        "architecture": "procedural",
        "success": bool(
            final["published"]
            or final["revision_requested"]
            or final["reason"] in {"invalid_request", "not_ready"}
        ),
        "outcome": final["reason"],
        "published": final["published"],
        "node_executions": final["node_executions"],
        "graph_transitions": pd.NA,
        "tool_calls": final["tool_calls"],
        "retries": final["retries"],
        "resumes": final["resumes"],
        "human_interventions": final["human_interventions"],
        "checkpoints": 0,
        "reconstructed_steps": final["reconstructed_steps"],
        "process_loss": process_loss,
    }

rows = []
for name in SCENARIOS:
    rows.append(procedural_metrics(name))
    rows.append(graph_metrics(name))

evidence = pd.DataFrame(rows)
display(evidence)

In [ ]:
observed = evidence.pivot(
    index="scenario",
    columns="architecture",
    values=[
        "success",
        "outcome",
        "node_executions",
        "tool_calls",
        "retries",
        "resumes",
        "human_interventions",
        "checkpoints",
        "reconstructed_steps",
    ],
)
display(observed)

### 10.1 Evidência primeiro; utility depois

A tabela anterior contém observações. Ela não sabe qual arquitetura sua organização prefere.

Para transformar medidas em uma decisão precisamos declarar **uma função de valor**.

Abaixo usamos dois perfis hipotéticos — aplicados com **a mesma fórmula às duas arquiteturas**:

```text
short_execution
→ overhead local importa mais

recovery_critical
→ reconstrução de progresso e intervenção operacional pesam mais
```

Os pesos são exemplos pedagógicos, não verdades universais. O ponto é justamente mostrar que:

> **utility depende do contexto; métricas não devem esconder preferências.**

In [ ]:
UTILITY_PROFILES = {
    "short_execution": {
        "node_executions": 1.00,
        "tool_calls": 0.50,
        "retries": 0.75,
        "resumes": 0.50,
        "human_interventions": 1.00,
        "checkpoints": 0.40,
        "reconstructed_steps": 1.00,
    },
    "recovery_critical": {
        "node_executions": 0.40,
        "tool_calls": 0.40,
        "retries": 0.75,
        "resumes": 0.50,
        "human_interventions": 1.50,
        "checkpoints": 0.20,
        "reconstructed_steps": 3.00,
    },
}

def burden_score(row, weights):
    return round(sum(float(row[k]) * w for k, w in weights.items()), 2)

utility_rows = []
for _, row in evidence.iterrows():
    for profile, weights in UTILITY_PROFILES.items():
        utility_rows.append({
            "scenario": row["scenario"],
            "architecture": row["architecture"],
            "utility_profile": profile,
            "burden_score_lower_is_better": burden_score(row, weights),
        })

utility = pd.DataFrame(utility_rows)
display(
    utility.pivot(
        index=["scenario", "utility_profile"],
        columns="architecture",
        values="burden_score_lower_is_better",
    )
)

### 10.2 Leitura honesta do experimento

O laboratório não demonstra que graph runtime é universalmente superior.

- **linear:** as duas arquiteturas chegam ao mesmo resultado; checkpointing tende a adicionar overhead.
- **branching:** agora há um desvio real para `request_revision`; ainda assim, código procedural continua perfeitamente viável.
- **transient_failure:** ambas isolam o retry em `publish`. Neste experimento pequeno, o grafo **não ganha** por isso; pode inclusive custar mais.
- **human_interrupt:** uma pausa curta no mesmo processo não exige durable execution.
- **durable_resume:** quando há perda do processo, o runtime com checkpoint retoma a partir do estado persistido, enquanto o baseline procedural precisa reconstruir progresso observável.

A lição não é “use grafos”. É:

> **coordenação explícita passa a conquistar sua complexidade quando persistência, retomada, recovery e supervisão criam valor mensurável no contexto.**

## 11. Failure Lab — checkpoint inválido

Persistência também cria uma nova superfície de falha.

Vamos simular um checkpoint que aponta para um node inexistente.

O objetivo é localizar a falha no runtime, não na decisão de negócio, e verificar uma distinção essencial:

```text
terminal
≠
successful

invalid runtime state
→ failed
```

In [ ]:
bad = make_state("linear")
bad.node = "node_that_does_not_exist"
save_checkpoint("bad-checkpoint", bad)

restored = load_checkpoint("bad-checkpoint")
result = run_graph(restored, "bad-checkpoint")

print("status:", result.status)
print("reason:", result.reason)
print("published:", result.published)

## 12. Do runtime determinístico ao agentic routing

Até aqui, a política de transição é determinística. Isso é intencional: primeiro isolamos a mecânica de orquestração.

Agora reconectamos a Aula 22 à Aula 20.

```text
deterministic routing
state → rule → next node

agentic routing
state → Decision Provider → proposed next node
                         ↓
                    governance
                         ↓
                    accepted route
```

O **Decision Provider não substitui o runtime**.

Ele propõe. O runtime continua responsável por validar a transição, persistir estado, impor limites, interromper, retomar e observar a execução.

In [ ]:
ALLOWED_TRANSITIONS = {
    "inspect": {"prepare", "request_revision"},
    "prepare": {"approval", "publish"},
    "approval": {"publish"},
    "publish": set(),
    "request_revision": set(),
}

def simulated_decision_provider(state):
    # Determinístico de propósito: queremos observar o contrato,
    # não introduzir variabilidade de LLM nesta aula.
    if state.node == "inspect":
        if state.reason == "needs_revision":
            return "request_revision"
        return "prepare"
    if state.node == "prepare":
        return "approval" if state.approval_required else "publish"
    if state.node == "approval" and state.approval_granted:
        return "publish"
    return None

def governed_agentic_route(state, proposal):
    allowed = ALLOWED_TRANSITIONS.get(state.node, set())
    if proposal is None:
        return None
    if proposal not in allowed:
        return None
    return proposal

demo = make_state("linear")
inspect_node(demo)

proposal = simulated_decision_provider(demo)
accepted = governed_agentic_route(demo, proposal)

print("proposal:", proposal)
print("accepted:", accepted)

unsafe_proposal = "publish"
print(
    "unsafe proposal accepted?",
    governed_agentic_route(demo, unsafe_proposal),
)

O exemplo mostra a separação de responsabilidades:

- o Decision Provider pode sugerir a próxima transição;
- a governança limita quais transições são admissíveis;
- o orchestration runtime continua sendo a camada de execução.

Isso evita uma confusão frequente:

> **um runtime de grafo pode ser totalmente determinístico; um sistema agente pode usar esse runtime sem entregar a ele a autoridade de decisão.**

## 13. Architecture Decision Lab

Escolha entre:

```text
procedural orchestration
graph runtime
```

Para cada caso, justifique por:

- quantidade de estados;
- branching;
- necessidade de persistência;
- necessidade de retomada;
- human-in-the-loop;
- duração do processo;
- reversibilidade;
- observabilidade;
- custo operacional;
- risco.

### Casos

**A.** Fluxo curto, linear, executado em segundos, sem aprovação humana.

**B.** Processo com vários branches, mas sempre concluído em uma única execução curta.

**C.** Processo que pode esperar horas por aprovação humana e deve continuar do ponto exato.

**D.** Processo longo, com falhas transitórias, checkpoints e necessidade de auditoria.

Pergunta central:

> **Em quais casos o graph runtime realmente conquistou sua complexidade?**

In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução de referência\n\n{self._solution_text}"))

q22_arch = TILExercise(
    "Branching, sozinho, não obriga um graph runtime. Observe principalmente persistência, retomada, duração e human-in-the-loop.",
    "A tende a procedural. B pode continuar procedural se o branching permanecer simples. C e D são os casos em que checkpoint, resume, recovery e auditabilidade começam a justificar um runtime explícito.",
)

print("Use q22_arch.hint() ou q22_arch.solution() somente quando desejar.")

### 13.1 Coding Lab — runtime sob pressão

Escolha um dos desafios.

#### Desafio A — checkpoint consistente

Explique por que o contador precisa ser atualizado **antes** do snapshot persistido. Em seguida, altere temporariamente a ordem em `save_checkpoint`, execute a demo e observe a inconsistência após `resume`.

#### Desafio B — compensação

Adicione um node `compensate` acionado quando uma publicação produz uma falha **não retryable**.

Perguntas:

1. por que compensation não é retry?
2. que estado precisa ser persistido antes da compensação?
3. a compensação pode falhar? Como o runtime deveria representar isso?

> O objetivo não é escrever mais código. É perceber como uma nova garantia operacional altera o contrato do runtime.

In [ ]:
q22_code = TILExercise(
    "Para compensação, pense em uma ação que desfaz ou neutraliza um efeito já produzido. Isso é diferente de repetir a mesma operação.",
    """Uma solução possível adiciona um estado de falha não retryable, roteia para compensate e registra explicitamente o resultado da compensação. O checkpoint deve preservar informação suficiente para saber qual side effect ocorreu e qual ação compensatória é admissível. Se a compensação falhar, o runtime deve terminar em failed (ou escalation_required), nunca fingir completed.""",
)

print("Use q22_code.hint() ou q22_code.solution() quando desejar.")

## 14. E onde entra LangGraph?

Somente agora.

LangGraph é uma implementação possível dos mecanismos que acabamos de construir.

Mapeamento conceitual:

```text
conceito TIL          → possível primitiva de framework

shared state          → state schema
node                  → node/function
transition policy     → edge / conditional edge
checkpoint            → checkpointer
interrupt             → interrupt
resume                → retomada a partir do estado persistido
```

A aula **não depende de LangGraph** para funcionar.

Isso preserva o princípio:

> **Primeiro o mecanismo; depois o framework.**

Se amanhã o framework mudar, os conceitos continuam válidos.

## 15. Checagem de compreensão

Responda antes de abrir as respostas de referência.

1. Qual a diferença entre Agent Harness e Orchestration Runtime?
2. O que torna um estado “shared state”?
3. Qual a diferença entre node e transition policy?
4. Por que checkpoint não é sinônimo de log?
5. Qual a diferença entre retry, resume e recovery?
6. O que durable execution acrescenta?
7. Por que branching sozinho não justifica um graph runtime?
8. Em que cenário o baseline procedural pode ser preferível?
9. Que nova superfície de falha nasce quando adicionamos persistência?
10. Por que LangGraph aparece somente no final da aula?
11. Que evidência justificaria migrar de código procedural para um runtime explícito?
12. Como esta aula estende o princípio “complexidade precisa ser conquistada por evidência”?

### Respostas de referência — revele depois de responder

<details>
<summary><strong>1. Agent Harness vs Orchestration Runtime</strong></summary>

O harness controla o comportamento agente ao redor da decisão — contexto, tools, políticas, limites e observação. O orchestration runtime coordena estado, nodes, transições, persistência, interrupção e retomada. Um runtime pode ser parte do harness.

</details>

<details>
<summary><strong>2. O que torna o estado compartilhado?</strong></summary>

Ele é a representação explícita de execução lida e atualizada por diferentes nodes e pelo próprio runtime, em vez de ficar escondido em variáveis locais de uma única função.

</details>

<details>
<summary><strong>3. Node vs transition policy</strong></summary>

Node transforma ou observa o estado. Transition policy escolhe qual node pode executar em seguida.

</details>

<details>
<summary><strong>4. Checkpoint vs log</strong></summary>

Log registra eventos. Checkpoint preserva estado suficiente para reconstruir uma execução válida e continuar a partir dele.

</details>

<details>
<summary><strong>5. Retry, resume e recovery</strong></summary>

Retry repete uma operação. Resume continua uma execução pausada a partir de estado preservado. Recovery restaura uma condição executável após falha e pode decidir retry, resume, compensação ou parada.

</details>

<details>
<summary><strong>6. Durable execution</strong></summary>

Permite que o progresso sobreviva à perda ou encerramento do processo original e seja retomado posteriormente.

</details>

<details>
<summary><strong>7. Por que branching não basta?</strong></summary>

Porque poucos branches podem ser coordenados com clareza por código procedural. O runtime precisa conquistar sua complexidade por necessidades adicionais como persistência, retomada, auditoria ou supervisão.

</details>

<details>
<summary><strong>8. Quando procedural pode ser preferível?</strong></summary>

Em fluxos curtos, estáveis, locais e facilmente observáveis, nos quais checkpointing e infraestrutura adicional não produzem utility suficiente.

</details>

<details>
<summary><strong>9. Nova superfície de falha da persistência</strong></summary>

Checkpoint obsoleto, corrompido, incompatível ou apontando para um estado/node inválido pode impedir uma retomada correta.

</details>

<details>
<summary><strong>10. Por que LangGraph aparece no final?</strong></summary>

Para que state, routing, checkpoint, interrupt e resume sejam compreendidos como mecanismos gerais antes de serem associados a uma implementação específica.

</details>

<details>
<summary><strong>11. Evidência para migrar</strong></summary>

Redução observável de reconstrução de progresso, melhor recovery, retomada durável, auditabilidade, coordenação ou segurança que compense o overhead conceitual e operacional.

</details>

<details>
<summary><strong>12. Como a aula aplica o princípio do TIL?</strong></summary>

Ela compara duas arquiteturas, separa métricas observadas de preferências de utility e só justifica a camada adicional quando capacidades concretas produzem valor no cenário.

</details>

## 16. Reprodutibilidade

```text
Internet OFF
GPU OFF
sem API externa
sem side effects reais
runtime local e determinístico
Decision Provider simulado
LangGraph opcional, não requerido
```

O laboratório principal depende apenas de Python e pandas disponíveis no ambiente do curso.

## 17. Síntese

```text
goal
→ shared state
→ node
→ governed transition
→ checkpoint
→ continue / interrupt / resume
→ recovery / compensate / fail
→ stop
```

A Aula 20 mostrou **como controlar um comportamento agente**.

A Aula 21 mostrou **como representar e validar um plano explícito**.

A Aula 22 mostra **como coordenar a execução ao longo do tempo e do estado** — e como um Decision Provider pode participar sem assumir o controle do runtime.

A principal conclusão experimental é deliberadamente condicional:

> **Um orchestration runtime não torna o sistema automaticamente melhor ou mais agente. Ele se justifica quando coordenação, persistência, retomada, recovery, auditabilidade ou supervisão produzem utility suficiente para pagar seu overhead.**

E a regra metodológica permanece:

> **evidência primeiro; função de utility explícita depois.**


---

## Continue no TIL

← **[Anterior: Aula 21 — Planning and Goal Decomposition](https://www.kaggle.com/code/pedrogentil/til-21-planning-and-goal-decomposition)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-applied-ai-engineering)**

TIL · Pedro Gentil Regato de Oliveira Soares · Conteúdo: CC BY 4.0 · Código: MIT
